# Task: Neural Network from Scratch

Build a 1-hidden-layer NN from scratch in NumPy that learns XOR; diagram the architecture.

### Architecture Diagram

```text
  [Input Layer]      [Hidden Layer]       [Output Layer]
     x1 (0) ------------> h1 (sigmoid) ---->
            \          /                    \
             \        /                      \
              \      /                        \
               \    /                          \
                \  /                            > y (sigmoid)
                 \/                            /
                 /\                           /
                /  \                         /
               /    \                       /
              /      \                     /
            /         \                   /
     x2 (0) ------------> h2 (sigmoid) ---->
```

- Input Layer: 2 neurons
- Hidden Layer: 2 neurons
- Output Layer: 1 neuron
- Activation: Sigmoid for both hidden and output layers
- Loss: Mean Squared Error (MSE)


In [ ]:
import numpy as np

# 1. Activation Functions
def sigmoid(x):
    # Clip inputs to prevent exponent overflow/underflow warnings in NumPy
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(output):
    # The derivative of sigmoid, expressed in terms of the output of the sigmoid function
    return output * (1 - output)

# 2. Dataset (XOR)
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([
    [0],
    [1],
    [1],
    [0]
])

# 3. Model Initialization
np.random.seed(42) # For reproducibility

input_layer_neurons = X.shape[1]
hidden_layer_neurons = 2
output_neurons = 1

# Weights and biases
weights_input_hidden = np.random.uniform(size=(input_layer_neurons, hidden_layer_neurons))
bias_hidden = np.zeros((1, hidden_layer_neurons))

weights_hidden_output = np.random.uniform(size=(hidden_layer_neurons, output_neurons))
bias_output = np.zeros((1, output_neurons))

learning_rate = 0.5
epochs = 10000

# 4. Training Loop
for epoch in range(epochs):
    # --- Forward Pass ---
    hidden_layer_input = np.dot(X, weights_input_hidden) + bias_hidden
    hidden_layer_activations = sigmoid(hidden_layer_input)
    
    output_layer_input = np.dot(hidden_layer_activations, weights_hidden_output) + bias_output
    predicted_output = sigmoid(output_layer_input)
    
    # Calculate loss
    loss = np.mean((y - predicted_output) ** 2)
    
    # --- Backward Pass (Backpropagation) ---
    # Error at output layer
    error = predicted_output - y
    
    # Gradients for output layer
    d_predicted_output = error * sigmoid_derivative(predicted_output)
    
    # Error at hidden layer
    error_hidden_layer = d_predicted_output.dot(weights_hidden_output.T)
    
    # Gradients for hidden layer
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_activations)
    
    # --- Update Weights and Biases ---
    weights_hidden_output -= hidden_layer_activations.T.dot(d_predicted_output) * learning_rate
    bias_output -= np.sum(d_predicted_output, axis=0, keepdims=True) * learning_rate
    
    weights_input_hidden -= X.T.dot(d_hidden_layer) * learning_rate
    bias_hidden -= np.sum(d_hidden_layer, axis=0, keepdims=True) * learning_rate
    
    # Logging
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# 5. Final Evaluation
print("\nTraining complete!")
print("Final Predictions (should be close to 0, 1, 1, 0):")
print(predicted_output)


Epoch 0, Loss: 0.2521


Epoch 2000, Loss: 0.0028


Epoch 4000, Loss: 0.0009


Epoch 6000, Loss: 0.0006


Epoch 8000, Loss: 0.0004



Training complete!
Final Predictions (should be close to 0, 1, 1, 0):
[[0.01903827]
 [0.98357372]
 [0.98356218]
 [0.01700765]]


### Edge and Error Case Validation

To comply with quality guidelines, we test the network with two key edge/error cases:
1. **Edge Case 1: Input shape validation.** We verify that the model raises a descriptive `ValueError` when passed input of incorrect shape/features.
2. **Edge Case 2: Numerical stability under extreme inputs.** By implementing a robust prediction pass with input clipping (or stable activation math), the network handles extremely large inputs without raising `RuntimeWarning` or generating `NaN` values.

In [ ]:
# 6. Edge and Error Case Handling

# We define a function for inference that explicitly validates inputs and prevents overflow.
def predict(X_in, w_ih, b_h, w_ho, b_o):
    # Edge/Error Case 1: Validate input shape and dimensions
    if X_in.ndim != 2:
        raise ValueError(f"Input must be 2-dimensional (samples, features), got {X_in.ndim}D.")
    if X_in.shape[1] != w_ih.shape[0]:
        raise ValueError(f"Feature dimension mismatch. Expected {w_ih.shape[0]} features, got {X_in.shape[1]}.")
    
    # Edge/Error Case 2: Numerical stability in activation
    # We clip inputs to avoid exponential overflow warnings in np.exp(-x)
    hidden_input = np.dot(X_in, w_ih) + b_h
    hidden_activations = sigmoid(hidden_input)
    
    output_input = np.dot(hidden_activations, w_ho) + b_o
    predictions = sigmoid(output_input)
    return predictions

# --- Test Edge Case 1: Incorrect input shape ---
try:
    print("Testing Edge Case 1 (mismatched features - 3 features instead of 2):")
    invalid_input = np.array([[1.0, 0.0, 1.0]])
    predict(invalid_input, weights_input_hidden, bias_hidden, weights_hidden_output, bias_output)
except ValueError as e:
    print(f"Successfully caught expected error: {e}\n")

# --- Test Edge Case 2: Extreme values (Numerical Stability) ---
print("Testing Edge Case 2 (extreme values that usually cause overflow):")
extreme_input = np.array([
    [1000.0, -1000.0],
    [-1000.0, 1000.0]
])
try:
    # This should execute cleanly without any RuntimeWarnings of overflow and yield stable predictions
    predictions_extreme = predict(extreme_input, weights_input_hidden, bias_hidden, weights_hidden_output, bias_output)
    print("Stable predictions for extreme inputs:")
    print(predictions_extreme)
except Exception as e:
    print(f"Failed to handle extreme values: {e}")


Testing Edge Case 1 (mismatched features - 3 features instead of 2):
Successfully caught expected error: Feature dimension mismatch. Expected 2 features, got 3.

Testing Edge Case 2 (extreme values that usually cause overflow):
Stable predictions for extreme inputs:
[[0.01156856]
 [0.99350439]]


### Reflections
- **What was difficult:** Getting the backpropagation right (matrix multiplications and transposes) so that the dimensions matched. Also, XOR is notoriously difficult without a hidden layer, so verifying that it works with just 2 hidden neurons was cool.
- **What I improved:** My understanding of how the error flows backwards through the network via the chain rule.
- **What remains:** Using modern frameworks (PyTorch/TensorFlow) which abstract all of this away!